### We provide an additional analysis on the limitation of current common MI evaluation framework by computing FP rates. The details can be found in the paper. 

1. Prepare the assessment of MI attacks by current common MI evaluation framework. It should have the same structure with the assessment of MI attacks using our MLLM-based MI evaluation framework. One example is `AttackSamples/CelebA/KEDMI/FFHQ/FaceNet64/prediction.csv`. In this Supplementary, we provide such files for 26 different MI setups in the `AttackSamples` directory

2. Run the cell below to compute some attribute accuracy.


In [ ]:
import os
from utility import *


base_path = "AttackSamples/Cifa100/PPA/Cifar10/Resnet18"   #Path to the MI folder that contains the ground truth and prediction csv files

ground_truth_path = os.path.join(base_path, "gemini_results_gemini-2.0-flash.csv")
prediction_path = os.path.join(base_path, "prediction.csv")

merged = compute_metrics(ground_truth_path, prediction_path)

In [ ]:
import pandas as pd

def compute_confusion_matrix_metrics(ground_truth_path, prediction_path):
    """
    Compute confusion matrix and metrics for MI evaluation.
    
    Args:
        ground_truth_path: Path to CSV file with ground truth (MLLM results)
        prediction_path: Path to CSV file with predictions (current method)
    
    Returns:
        Dictionary with all computed metrics
    """
    # Read the CSV files
    ground_truth = pd.read_csv(ground_truth_path)
    predictions = pd.read_csv(prediction_path)
    
    # Ensure the column names are consistent
    ground_truth.columns = ['filename', 'label']
    predictions.columns = ['filename', 'prediction']
    
    # Merge the dataframes on filename
    merged = pd.merge(ground_truth, predictions, on='filename', how='inner')
    
    print(f"Total matched files: {len(merged)}")
    
    # Convert labels to binary (yes=1, no=0)
    merged['label'] = (merged['label'].str.lower() == 'yes').astype(int)
    merged['prediction'] = (merged['prediction'].str.lower() == 'yes').astype(int)
    
    # Calculate confusion matrix
    TP = ((merged['label'] == 1) & (merged['prediction'] == 1)).sum()
    TN = ((merged['label'] == 0) & (merged['prediction'] == 0)).sum()
    FP = ((merged['label'] == 0) & (merged['prediction'] == 1)).sum()
    FN = ((merged['label'] == 1) & (merged['prediction'] == 0)).sum()
    
    # Calculate rates
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0  # True Positive Rate (Sensitivity)
    TNR = TN / (TN + FP) if (TN + FP) > 0 else 0  # True Negative Rate (Specificity)
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0  # False Positive Rate
    FNR = FN / (FN + TP) if (FN + TP) > 0 else 0  # False Negative Rate
    
    # Calculate Attribute Accuracies
    AttAcc_F_MLLM = (merged['label'] == 1).sum() / len(merged) if len(merged) > 0 else 0
    AttAcc_F_Curr = (merged['prediction'] == 1).sum() / len(merged) if len(merged) > 0 else 0
    
    # Print results
    print("\n" + "="*60)
    print("Confusion Matrix Metrics:")
    print("="*60)
    print(f"TP (True Positives):  {TP}")
    print(f"TN (True Negatives):  {TN}")
    print(f"FP (False Positives): {FP}")
    print(f"FN (False Negatives): {FN}")
    
    print("\n" + "="*60)
    print("Attribute Accuracy:")
    print("="*60)
    print(f"AttAcc_F_MLLM: {AttAcc_F_MLLM*100:.2f}%")
    print(f"AttAcc_F_Curr: {AttAcc_F_Curr*100:.2f}%")
    
    print("\n" + "="*60)
    print("Rates:")
    print("="*60)
    print(f"FPR (False Positive Rate): {FPR*100:.2f}%")
    print(f"FNR (False Negative Rate): {FNR*100:.2f}%")
    print(f"TPR (True Positive Rate):  {TPR*100:.2f}%")
    print(f"TNR (True Negative Rate):  {TNR*100:.2f}%")
    
    # Return metrics as dictionary
    return {
        'TP': TP,
        'TN': TN,
        'FP': FP,
        'FN': FN,
        'AttAcc_F_MLLM': AttAcc_F_MLLM,
        'AttAcc_F_Curr': AttAcc_F_Curr,
        'FPR': FPR,
        'FNR': FNR,
        'TPR': TPR,
        'TNR': TNR
    }

# Example usage for CIFAR100
base_path = "AttackSamples/Cifa100/PPA/Cifar10/Resnet18"
ground_truth_path = os.path.join(base_path, "gemini_results_gemini-2.0-flash.csv")
prediction_path = os.path.join(base_path, "prediction.csv")

metrics = compute_confusion_matrix_metrics(ground_truth_path, prediction_path)

